> **Solutions for `5-linear-algebra-lab.ipynb`.** Work through the lab on your own first. Open this notebook to check your answers after you have made a real attempt, or when you have been stuck on a part for a while. Many questions have more than one good answer, so compare your reasoning, not just your wording.

# Lab 5: Linear Algebra Essentials

> **Course:** Data Science Foundations
> **Lesson:** Day 5 Lab, companion to `5-linear-algebra-essentials.md`
> **Dataset:** `mpg` (seaborn), reused from Day 1 so the focus stays on the maths
> **Estimated time:** 2 hours

## Lab Objectives

- Do vector operations by hand, then verify them in NumPy, including angles and orthogonality
- Implement matrix multiplication with loops and confirm the dimension rule
- Check key matrix properties: non-commutativity, the transpose of a product, the identity, the **inverse**, and singular matrices
- Use **cosine similarity** on standardised data, and see why raw dot products mislead
- Build a covariance matrix with one matrix product
- Fit linear regression with the normal equations the right way: split before scaling, and `solve` instead of `inv`
- See how **collinear** features break $X^TX$

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

cars = sns.load_dataset("mpg").dropna().reset_index(drop=True)
print(f"Shape: {cars.shape}")

## Part 1: Vectors by Hand

$$\vec{u} = \begin{bmatrix} 3 \\ -1 \\ 2 \end{bmatrix}, \qquad \vec{v} = \begin{bmatrix} 1 \\ 4 \\ 0 \end{bmatrix}, \qquad \vec{w} = \begin{bmatrix} 2 \\ 2 \\ -2 \end{bmatrix}$$

Work these out **on paper first**, then type your answers as plain numbers (no NumPy) in the cell below. The checkpoint compares them with NumPy.

In [ ]:
by_hand = {
    "u + v": [4, 3, 2],
    "2u - v": [5, -6, 4],
    "u . v": 3 * 1 + (-1) * 4 + 2 * 0,     # = -1
    "||u||": 14 ** 0.5,
    "u . w": 3 * 2 + (-1) * 2 + 2 * (-2),  # = 0
}

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
u, v, w = np.array([3, -1, 2]), np.array([1, 4, 0]), np.array([2, 2, -2])
assert np.allclose(by_hand["u + v"], u + v)
assert np.allclose(by_hand["2u - v"], 2 * u - v)
assert np.isclose(by_hand["u . v"], u @ v)
assert np.isclose(by_hand["||u||"], np.linalg.norm(u))
assert np.isclose(by_hand["u . w"], u @ w)
print("passed")

In [ ]:
cos_theta = (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))
angle_uv = np.degrees(np.arccos(cos_theta))
print(f"cos(theta) = {cos_theta:.4f}, angle = {angle_uv:.1f} degrees")

**Question 1.** What does $\vec{u} \cdot \vec{w} = 0$ mean geometrically? Why is $\vec{u}$ and $\vec{v}$ being almost perpendicular (about 94°) consistent with a dot product close to 0?

**Model answer:**

A zero dot product means $\vec{u}$ and $\vec{w}$ are **orthogonal** (at 90°): neither has any component in the other's direction. $\vec{u} \cdot \vec{v} = -1$ is small compared with $\|\vec{u}\|\|\vec{v}\| \approx 15.4$, so $\cos\theta \approx -0.065$ and the angle is just over 90°. The *sign* says they point very slightly apart; the *size relative to the norms* says they are nearly unrelated.

## Part 2: Matrix Multiplication with Loops

Each entry of $C = AB$ is a dot product: $C_{ij} = \sum_k A_{ik} B_{kj}$.

In [ ]:
def my_matmul(A, B):
    A, B = np.asarray(A, dtype=float), np.asarray(B, dtype=float)
    m, n = A.shape
    n2, p = B.shape
    if n != n2:
        raise ValueError(f"Inner dimensions do not match: {A.shape} @ {B.shape}")
    C = np.zeros((m, p))
    for i in range(m):
        for j in range(p):
            total = 0.0
            for k in range(n):
                total += A[i, k] * B[k, j]
            C[i, j] = total
    return C

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(my_matmul([[1, 2], [3, 4]], [[5, 6], [7, 8]]), [[19, 22], [43, 50]])   # lesson example
for shape_a, shape_b in [((3, 4), (4, 2)), ((1, 5), (5, 1)), ((5, 1), (1, 5))]:
    A, B = rng.normal(size=shape_a), rng.normal(size=shape_b)
    assert np.allclose(my_matmul(A, B), A @ B)
try:
    my_matmul(np.ones((2, 3)), np.ones((2, 3)))
    raise AssertionError("should have raised ValueError")
except ValueError:
    pass
print("passed")

In [ ]:
A, B = rng.normal(size=(120, 120)), rng.normal(size=(120, 120))
t0 = time.perf_counter(); my_matmul(A, B); t_loops = time.perf_counter() - t0
t0 = time.perf_counter(); A @ B; t_numpy = time.perf_counter() - t0
print(f"Loops: {t_loops:.3f} s   NumPy @: {t_numpy:.5f} s   (about {t_loops / t_numpy:,.0f}x faster)")

**Question 2.** A (1×5) @ (5×1) and a (5×1) @ (1×5) product use the same numbers. What shapes do they produce, and what are they called?

**Model answer:**

(1×5)(5×1) is **1×1**: a single number, the dot product (the *inner* product). (5×1)(1×5) is **5×5**: every pairwise product $a_i b_j$, called the *outer* product. The same two vectors give completely different results depending on order, which is the most dramatic example of $AB \ne BA$.

## Part 3: Matrix Properties and the Inverse

The lesson introduced the inverse briefly. Here you will work with it directly. The **inverse** $A^{-1}$ of a square matrix $A$ satisfies $A A^{-1} = A^{-1} A = I$. It exists only when $A$ is **non-singular**: its determinant is non-zero, or equivalently its columns are **linearly independent** (no column is a combination of the others).

In [ ]:
A = np.array([[2, 1, 0], [1, 3, 1], [0, 1, 4]], dtype=float)
B = np.array([[1, 2, 0], [0, 1, 1], [1, 0, 2]], dtype=float)
I = np.eye(3)

ab_equals_ba = np.allclose(A @ B, B @ A)
transpose_rule = np.allclose((A @ B).T, B.T @ A.T)
identity_rule = np.allclose(A @ I, A)
print(f"AB == BA: {ab_equals_ba}   (AB)^T == B^T A^T: {transpose_rule}   AI == A: {identity_rule}")

A_inv = np.linalg.inv(A)
print("A @ A_inv =\n", A @ A_inv)
print("Is identity:", np.allclose(A @ A_inv, I))

S = np.array([[1, 2], [2, 4]], dtype=float)
print(f"det(S) = {np.linalg.det(S):.1f}, rank(S) = {np.linalg.matrix_rank(S)}")
try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as err:
    print("np.linalg.inv(S) failed:", err)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert ab_equals_ba is not None and not ab_equals_ba
assert transpose_rule and identity_rule
assert np.allclose(A @ A_inv, np.eye(3))
print("passed")

### 3.4 Solving a system without the inverse

$$2x + y = 5, \qquad x + 3y = 10 \quad \Longleftrightarrow \quad \begin{bmatrix} 2 & 1 \\ 1 & 3 \end{bmatrix}\begin{bmatrix} x \\ y \end{bmatrix} = \begin{bmatrix} 5 \\ 10 \end{bmatrix}$$

`np.linalg.solve(M, b)` finds the solution directly. It is faster and more accurate than `np.linalg.inv(M) @ b`, which is why numerical code avoids computing inverses.

In [ ]:
M = np.array([[2, 1], [1, 3]], dtype=float)
b = np.array([5, 10], dtype=float)
xy = np.linalg.solve(M, b)
print(f"x = {xy[0]:.1f}, y = {xy[1]:.1f}")
print("Check M @ xy =", M @ xy)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(xy, [1, 3])
print("passed")

**Question 3.** Why can't $S$ be inverted? Describe it in terms of its columns.

**Model answer:**

The second column of $S$ is exactly 2 × the first, so the columns are linearly dependent. $S$ squashes every 2-D vector onto a single line (its rank is 1), and information is lost. Different inputs such as $(2, 0)$ and $(0, 1)$ map to the same output $(2, 4)$, so no matrix can undo it. The determinant is 0, which measures exactly this collapse of area to nothing.

## Part 4: Cosine Similarity Between Cars

The lesson compared houses using cosine similarity on standardised features. Here you will see what goes wrong without those two steps. A **raw** dot product favours rows with big numbers, not similar rows. **Cosine similarity** divides out the lengths and keeps only direction:

$$\cos(\theta) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\|\,\|\vec{b}\|}$$

On standardised data it asks: *is this car above or below average in the same features as the target car?*

In [ ]:
features = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration"]
X_raw = cars[features].to_numpy(dtype=float)
target_idx = cars.index[cars["name"] == "ford pinto"][0]
print(f"Target car: row {target_idx}")
cars.loc[target_idx, ["name"] + features]

In [ ]:
def cosine_similarity(a, B):
    return (B @ a) / (np.linalg.norm(B, axis=1) * np.linalg.norm(a))


X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)

raw_scores = X_raw @ X_raw[target_idx]
cos_scores = cosine_similarity(X_std[target_idx], X_std)

def top5(scores):
    order = np.argsort(scores)[::-1]
    order = order[order != target_idx][:5]
    return cars.loc[order, ["name", "mpg", "cylinders", "weight"]]

print("Most 'similar' by raw dot product:")
display(top5(raw_scores))
print("Most similar by cosine similarity on standardised features:")
display(top5(cos_scores))

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.isclose(cosine_similarity(np.array([1.0, 0]), np.array([[2.0, 0], [0, 3.0]]))[0], 1)
assert np.isclose(cosine_similarity(np.array([1.0, 0]), np.array([[2.0, 0], [0, 3.0]]))[1], 0)
print("passed")

**Question 4.** Compare the two lists. Why does the raw dot product pick those cars?

**Model answer:**

The raw dot product returns heavy 8-cylinder American cars, nothing like a Pinto. `weight` (thousands) and `displacement` (hundreds) dominate the sum, so the largest scores simply go to the heaviest, biggest-engined cars. It measures "how big are both vectors", not "how alike".

Cosine similarity on standardised features returns small 4-cylinder economy cars with similar mpg and weight, which is what a person would call similar. Two fixes are at work: **standardising** puts every feature on the same scale, and **dividing by the norms** removes the effect of overall magnitude.

## Part 5: The Covariance Matrix Is One Matrix Product

If $X_c$ is the data with each column's mean subtracted (centred), then

$$\text{Cov} = \frac{1}{n - 1} X_c^T X_c$$

Entry $(i, j)$ is the dot product of centred column $i$ with centred column $j$, divided by $n-1$: exactly Monday's covariance formula.

In [ ]:
X_c = X_raw - X_raw.mean(axis=0)
cov_matrix = X_c.T @ X_c / (len(X_raw) - 1)
print("Matches pandas:", np.allclose(cov_matrix, cars[features].cov().to_numpy()))
pd.DataFrame(cov_matrix, index=features, columns=features).round(1)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(cov_matrix, cars[features].cov().to_numpy())
assert np.allclose(cov_matrix, cov_matrix.T)
print("passed: and the matrix is symmetric, as the lesson said")

## Part 6: Normal Equations, Done Properly

Predict `mpg` from six features. Keep the two habits from the lesson:

1. **Split first, then scale.** If you fit `StandardScaler` on *all* the data before splitting, the test set's mean and standard deviation leak into training. Fit the scaler on the training set only, then apply it to both sets.
2. **Solve, don't invert.** Use `np.linalg.solve(XtX, Xty)` instead of `np.linalg.inv(XtX) @ Xty`.

In [ ]:
predictors = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
X = cars[predictors].to_numpy(dtype=float)
y = cars["mpg"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)          # learn mean/std from training data only
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)


def add_intercept(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])


Xd_train, Xd_test = add_intercept(X_train_s), add_intercept(X_test_s)

w = np.linalg.solve(Xd_train.T @ Xd_train, Xd_train.T @ y_train)
for name, weight in zip(["intercept"] + predictors, w):
    print(f"{name:<13s} {weight:+.3f}")

In [ ]:
w_lstsq = np.linalg.lstsq(Xd_train, y_train, rcond=None)[0]
sk = LinearRegression().fit(X_train_s, y_train)
w_sklearn = np.concatenate([[sk.intercept_], sk.coef_])
print("max |w - lstsq|  :", np.abs(w - w_lstsq).max())
print("max |w - sklearn|:", np.abs(w - w_sklearn).max())

y_pred = Xd_test @ w
print(f"Test R^2 = {r2_score(y_test, y_pred):.3f}, RMSE = {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} mpg")

residuals = y_train - Xd_train @ w
print("X^T r =", Xd_train.T @ residuals)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert np.allclose(w, w_lstsq) and np.allclose(w, w_sklearn)
assert np.allclose(Xd_train.T @ (y_train - Xd_train @ w), 0, atol=1e-8)
print("passed")

**Question 6.** Which standardised coefficients are largest in magnitude? `horsepower` and `displacement` get tiny weights (displacement's is even positive), although both are strongly negatively correlated with mpg. How can that be?

**Model answer:**

`weight` (large negative) and `model_year` (large positive) dominate. Heavier cars use more fuel, and newer cars in this 1970 to 1982 period were more efficient.

`horsepower`, `displacement` and `cylinders` are all highly correlated with `weight` and with each other. A regression coefficient measures a feature's effect *while holding the other features fixed*. Once weight is in the model, horsepower and displacement have little extra to explain, so their coefficients shrink towards zero and can even flip sign. Individual coefficients of collinear features are unstable and should not be read as "this feature's effect". Part 7 shows the extreme version.

## Part 7: When $X^TX$ Breaks: Collinearity

The lesson mentions that $X^TX$ can be singular "with collinear features". Watch it happen: add `weight` again, converted to kilograms. It is the same information, so the new column is an exact multiple of an existing one.

In [ ]:
X_train_bad = np.column_stack([X_train, X_train[:, predictors.index("weight")] * 0.4536])
Xd_bad = add_intercept(StandardScaler().fit_transform(X_train_bad))

print(f"columns = {Xd_bad.shape[1]}, rank = {np.linalg.matrix_rank(Xd_bad)}")
print(f"condition number of X^T X: {np.linalg.cond(Xd_bad.T @ Xd_bad):.3e}")

try:
    w_bad = np.linalg.solve(Xd_bad.T @ Xd_bad, Xd_bad.T @ y_train)
    print("solve() returned weights (numerically meaningless):", w_bad.round(1))
except np.linalg.LinAlgError as err:
    print("solve() failed:", err)

w_min_norm = np.linalg.lstsq(Xd_bad, y_train, rcond=None)[0]
print("lstsq weights:", w_min_norm.round(3))
print(f"weight: {w_min_norm[4]:+.3f}, weight_kg: {w_min_norm[-1]:+.3f}, sum = {w_min_norm[4] + w_min_norm[-1]:+.3f}  (good model: {w[4]:+.3f})")
print("Same training predictions as the good model:", np.allclose(Xd_bad @ w_min_norm, Xd_train @ w))

**Question 7.** Explain what you saw. Why can't the data decide how to split the weight effect between the two columns?

**Model answer:**

The design matrix has 8 columns but rank 7, so $X^TX$ is singular: its condition number is astronomically large. `solve` either raises "Singular matrix" or, as here, returns an arbitrary split of the weight effect (about −1.7 and −3.8) that is decided by rounding error, not by the data.

After standardising, `weight` and `weight_kg` are the *same* column. Any split $a \cdot \text{weight} + b \cdot \text{weight\_kg}$ with the same $a + b$ gives identical predictions, so infinitely many weight vectors fit equally well and the normal equations have no unique solution. `lstsq` picks the smallest-norm one, splitting the effect evenly. Predictions are fine, but the individual coefficients are arbitrary. Real data is rarely *exactly* collinear, but near-collinearity (Part 6) causes the same instability in a milder form. Regularisation (ridge regression), which you will meet later, is the standard fix.

## Part 8: Wrap-Up

In 3 to 4 sentences, explain to a classmate why "multiply the data matrix by a weight vector" is the same thing as "compute a dot product for every row", and where $X^T$ shows up when fitting the model.

**Model answer:**

$X\vec{w}$ takes each row of $X$ (one car's features) and dots it with $\vec{w}$, producing one prediction per row. A matrix-vector product is just a stack of dot products, computed all at once. When fitting, we need the residual to be orthogonal to every *column* of $X$ (every feature direction), and "dot the residual with every column" is exactly $X^T\vec{r} = 0$. Rearranging that gives the normal equations $X^TX\vec{w} = X^T\vec{y}$.

> Submit your completed notebook to the Kanban board under your name by end of day.